In [1]:
!pip install mlflow scikit-learn pandas flask


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!wget --no-verbose https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv -O data/WA_Fn-UseC_-Telco-Customer-Churn.csv


2026-10-22 19:58:06 URL:https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv [970457/970457] -> "data/WA_Fn-UseC_-Telco-Customer-Churn.csv" [1]


In [3]:
# Load and Explore Data
import pandas as pd

# Load data
data = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Display the first few rows
print(data.head())

# Check for missing values
print(data.isnull().sum())

# Get basic info about the data
print(data.info())

# Describe the data
print(data.describe())


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [4]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# 1. Load and preprocess data
def load_and_preprocess_data(data_path):
    data = pd.read_csv(data_path)

    # Drop unnecessary identifier columns
    data = data.drop(['customerID'], axis=1)

    # Convert TotalCharges to numeric, coercing errors to NaN
    data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
    data['TotalCharges'].fillna(data['TotalCharges'].mean(), inplace=True)

    # Encode target variable
    label_encoder = LabelEncoder()
    data['Churn'] = label_encoder.fit_transform(data['Churn'])

    # One-hot encode categorical features
    categorical_cols = data.select_dtypes(include=['object']).columns
    data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

    return data

# 2. Split into features (X) and target (y)
def split_data(data, target_column):
    X = data.drop(target_column, axis=1)
    y = data[target_column]
    return X, y

if __name__ == "__main__":
    data_path = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

    if not os.path.isfile(data_path):
        print(f"Error: Data file not found at {data_path}")
        exit()

    try:
        data = load_and_preprocess_data(data_path)
        X, y = split_data(data, "Churn")
    except Exception as e:
        print(f"Error processing data: {e}")
        exit()

    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale features while preserving DataFrame column names
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    # Dedicated experiment name to isolate from other runs
    mlflow.set_experiment("Telco_Customer_Churn_Experimentation")

    with mlflow.start_run():
        params = {
            "penalty": "l2",
            "C": 0.1,
            "solver": "liblinear",
            "random_state": 42
        }

        mlflow.log_params(params)

        # Train model
        model = LogisticRegression(**params)
        model.fit(X_train_scaled, y_train)

        # Predict & Evaluate
        y_pred = model.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        # Log model WITH input_example so MLflow saves the 30-feature schema signature
        mlflow.sklearn.log_model(
            sk_model=model, 
            artifact_path="model",
            input_example=X_train_scaled.iloc[:1]
        )

        print("✅ Telco Model trained and logged successfully to MLflow!")
        print(f"Model accuracy:  {accuracy:.4f}")
        print(f"Model precision: {precision:.4f}")
        print(f"Model recall:    {recall:.4f}")
        print(f"Model f1-score:  {f1:.4f}")


2026/10/22 20:00:13 INFO mlflow.tracking.fluent: Experiment with name 'Telco_Customer_Churn_Experimentation' does not exist. Creating a new experiment.
/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/mlflow/types/utils.py:393: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model s

✅ Telco Model trained and logged successfully to MLflow!
Model accuracy:  0.8211
Model precision: 0.6873
Model recall:    0.5952
Model f1-score:  0.6379


In [5]:
from flask import Flask, request, jsonify
import mlflow.pyfunc
import pandas as pd
import os
import threading
import socket

app = Flask(__name__)

# Get the latest logged model from MLflow
def get_latest_model():
    mlflow_tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
    mlflow.set_tracking_uri(mlflow_tracking_uri)

    try:
        # Fetch specifically from the Telco experiment
        exp = mlflow.get_experiment_by_name("Telco_Customer_Churn_Experimentation")

        runs = mlflow.search_runs(
            experiment_ids=[exp.experiment_id] if exp else None,
            filter_string="metrics.accuracy > 0.5", 
            order_by=["attribute.start_time DESC"],
            max_results=1
        )

        if runs.empty:
            print("⚠️ No runs found matching criteria.")
            return None

        run_id = runs.iloc[0]["run_id"]
        model_uri = f"runs:/{run_id}/model"
        print(f"✅ Loading Telco model from Run ID: {run_id}")
        return mlflow.pyfunc.load_model(model_uri)
    except Exception as e:
        print(f"⚠️ Error fetching model from MLflow: {e}")
        return None

model = get_latest_model()

@app.route("/predict", methods=["POST"])
def predict():
    if model is None:
        return jsonify({"error": "No valid model loaded from MLflow."}), 500

    try:
        data = request.get_json()
        df = pd.DataFrame(data)

        # 1. Clean TotalCharges numeric type
        if "TotalCharges" in df.columns:
            df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

        # 2. One-hot encode categorical strings
        categorical_cols = df.select_dtypes(include=["object"]).columns
        df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

        # 3. Retrieve expected features & align schema
        try:
            expected_features = model.metadata.get_input_schema().input_names()
            df = df.reindex(columns=expected_features, fill_value=0)
        except Exception:
            pass  # Fallback if signature isn't present

        # 4. Cast all columns to float64 to match MLflow's double schema enforcement
        df = df.astype(float)

        # 5. Make prediction
        predictions = model.predict(df)
        return jsonify({"predictions": predictions.tolist()})

    except Exception as e:
        return jsonify({"error": str(e)}), 400

# Prevent thread locking on port 5011
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if __name__ == "__main__":
    if not is_port_in_use(5011):
        # Run Flask in a background thread with debug/reloader disabled
        threading.Thread(
            target=lambda: app.run(host="0.0.0.0", port=5011, debug=False, use_reloader=False), 
            daemon=True
        ).start()
        print("🚀 Flask deployment server running on http://127.0.0.1:5011")
    else:
        print("⚡ Server is already running on http://127.0.0.1:5011")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loading Telco model from Run ID: bd894429b9d94783ac4fc57aa04077bc


⚡ Server is already running on http://127.0.0.1:5011


In [6]:
!curl -X POST -H "Content-Type: application/json" -d '[{"SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No", "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service", "OnlineSecurity": "No", "OnlineBackup": "Yes", "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No", "StreamingMovies": "No", "PaperlessBilling": "Yes", "MonthlyCharges": 29.85, "TotalCharges": 29.85, "InternetService": "DSL", "Contract": "Month-to-month", "PaymentMethod": "Electronic check"}]' http://127.0.0.1:5011/predict


{"error":"X has 4 features, but LogisticRegression is expecting 30 features as input."}
